# DeepSDF: Gaussian vs Diffusion in the Small-Data Regime

Colab driver for the final project. This notebook is a **thin wrapper**: it clones the repo, installs dependencies, mounts Drive for persistence, and calls `scripts/run_grid.py` and `scripts/make_figures.py`. All logic lives in `src/`.

Recommended runtime: **GPU (L4 / A100 on Colab Pro)**.

## 1. Setup: clone, install, mount Drive

In [ ]:
# Clone the repository (edit the URL to your remote).
# !git clone https://github.com/<you>/<repo>.git
# %cd <repo>/Final_Project/code
!pip install -r requirements.txt

In [ ]:
# Mount Google Drive so checkpoints / outputs survive session restarts.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/deepsdf_generative/outputs/grid'
FIG_DIR = '/content/drive/MyDrive/deepsdf_generative/figures'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Run the N x D grid

`configs/base.yaml` runs the full sweep on GPU. Use `configs/smoke.yaml` for a quick CPU check. The run is resumable: each cell checkpoints to Drive.

In [ ]:
!python scripts/run_grid.py --config configs/base.yaml --output "$OUTPUT_DIR"

## 3. Regenerate tables and figures for the report

In [ ]:
!python scripts/make_figures.py --input "$OUTPUT_DIR" --figures "$FIG_DIR"

from IPython.display import Image, display
for name in ['degradation_generation.png', 'degradation_reconstruction.png', 'loss_curves.png', 'gallery.png']:
    display(Image(filename=f'{FIG_DIR}/{name}'))

## 4. (Optional) Use a real ShapeNet chairs subset

Upload a folder of chair meshes to Drive, then edit `configs/base.yaml`:

```yaml
data:
  source: mesh_dir
  mesh_dir: /content/drive/MyDrive/shapenet_chairs
```

Inspect the directory first:

In [ ]:
# !python scripts/prepare_data.py --inspect-dir /content/drive/MyDrive/shapenet_chairs